# 07 — PSM baseline: covariáveis socioeconômicas

Constrói a tabela de covariáveis pré-tratamento para o **Propensity Score Matching** (PSM) declarado em §3.7 do pré-registro v2.2.

**Decisões metodológicas:**
- P1: ano de referência uniforme = 2017 (Censo Agro).
- P2: drop colunas com >50% missing (`16_populacao_atentida_esgoto`, `15_area_irrig_ha_arroz`); imputa 0 em irrigação detalhada e área de cana (semântica de ausência).
- P3: derivar 7 covariáveis sintéticas (densidade pop, share VAB agro, tratores per estab, etc.).
- P4: filtro CS via `0_sg_uf ∈ {SP, MG, GO, MS, MT, PR}`.
- P5: renomear colunas (165 → 58 canônicas + dropar 105 secundárias).

**Outputs em `data/interim/`:**
- `psm_baseline_clean.csv` — 2.363 munis × 71 covariáveis (58 renomeadas + 7 derivadas + 6 dummies bioma).

**Outputs em `outputs_pre/`:**
- `psm_columns_provenance.csv` — mapeamento original → canônico para auditoria.

**Tempo esperado:** ~15 segundos.

In [1]:
# Setup portável — resolve a raiz do repositório sem depender do Google Drive.
# Para executar a partir do Drive, defina antes: os.environ["RENOVABIO_BASE_DIR"] = "<caminho>"
# Para executar a partir do Drive, defina antes de rodar esta célula:
import os
import sys
from pathlib import Path

if os.environ.get("RENOVABIO_BASE_DIR"):
    BASE_DIR = Path(os.environ["RENOVABIO_BASE_DIR"]).expanduser().resolve()
else:
    BASE_DIR = Path.cwd().resolve()
    while not (BASE_DIR / "requirements.txt").exists() and BASE_DIR != BASE_DIR.parent:
        BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [2]:
# Reload módulos
import importlib
from pipeline import config, normalize, psm_baseline
importlib.reload(config); importlib.reload(normalize); importlib.reload(psm_baseline)

from pipeline.config import PARAMS, interim, out_pre
from pipeline.psm_baseline import (
    run_psm_baseline_pipeline,
    COLUMN_RENAME, BIOMA_FIXES,
)
print('✓ módulos carregados')
print(f'  COLUMN_RENAME: {len(COLUMN_RENAME)} mapeamentos canônicos')
print(f'  BIOMA_FIXES: {len(BIOMA_FIXES)} correções de mojibake')

✓ módulos carregados
  COLUMN_RENAME: 58 mapeamentos canônicos
  BIOMA_FIXES: 6 correções de mojibake


## Roda pipeline PSM baseline

In [3]:
result = run_psm_baseline_pipeline(save=True)

→ Lendo base PSM raw...
  raw: (5570, 165)

→ Corrigindo mojibake do bioma...

→ Descartando colunas com >50% missing...
  após drop: (5570, 163)

→ Renomeando colunas (165 → ~50 canônicas)...
  renomeadas: 58 | descartadas: 105

→ Tipagem (geocode str, números float)...

→ Imputação de 0 em colunas com semântica de ausência...

→ Derivando 7 covariáveis sintéticas...

→ Adicionando dummies de bioma...
  dummies criadas: ['bioma_Amazônia', 'bioma_Caatinga', 'bioma_Cerrado', 'bioma_Mata Atlântica', 'bioma_Pampa', 'bioma_Pantanal']

→ Filtrando ao Centro-Sul (6 UFs)...
  CS: 2363 munis (esperado 2.363)
  Por UF:
uf
GO    246
MG    853
MS     79
MT    141
PR    399
SP    645

→ Salvando...
  ✓ tudo salvo


## Inspeção do output

In [4]:
data = result['data']
print(f'Shape: {data.shape}')
print(f'\nTipos das colunas:')
print(data.dtypes.value_counts().to_string())
print(f'\nColunas (71):')
for i, c in enumerate(data.columns, 1):
    print(f'  {i:2d}. {c}')

Shape: (2363, 71)

Tipos das colunas:
float64    51
Int64       8
int64       8
object      4

Colunas (71):
   1. uf
   2. geocode
   3. municipio
   4. is_amazonia_legal
   5. is_semiarido
   6. vab_agro
   7. vab_industria
   8. vab_servicos
   9. vab_admin_publica
  10. vab_bruto
  11. pib_total
  12. pib_percap
  13. pop_2017
  14. mb_share_agric_total_pre
  15. mb_share_cana_pre
  16. mb_share_pasto_pre
  17. mb_share_silvic_pre
  18. mb_share_soja_pre
  19. mb_share_vegnat_pre
  20. pam_area_colhida_total
  21. pam_area_cana
  22. pam_area_milho
  23. pam_area_soja
  24. pam_qtd_cana
  25. pam_val_total
  26. pam_val_cana
  27. censo_n_estab_total
  28. censo_n_estab_af
  29. censo_n_estab_mp
  30. censo_n_estab_lavperm
  31. censo_n_estab_lavtemp
  32. censo_area_lavoura
  33. censo_area_pecuaria
  34. censo_n_estab_com_energia
  35. censo_area_estab_total
  36. censo_n_pess_ocup
  37. censo_n_estab_com_trator
  38. censo_n_tratores
  39. censo_n_estab_irrig
  40. censo_area_ir

In [5]:
# Validação: distribuição por UF
print(f'Munis por UF:')
print(data.groupby("uf").size().to_string())
print()
print(f'Biomas (após fix mojibake):')
print(data['bioma'].value_counts(dropna=False).to_string())
print()
print(f'Top 10 missing values por coluna:')
print(data.isna().sum().sort_values(ascending=False).head(10).to_string())

Munis por UF:
uf
GO    246
MG    853
MS     79
MT    141
PR    399
SP    645

Biomas (após fix mojibake):
bioma
Mata Atlântica    1335
Cerrado            955
Amazônia            41
Pantanal            22
Caatinga            10

Top 10 missing values por coluna:
pop_atendida_agua         122
pop_urbana                122
pam_area_colhida_total     38
pam_val_total              38
share_area_irrig            9
censo_n_pess_ocup           6
censo_n_estab_total         5
share_estab_af              5
censo_n_estab_mp            5
censo_area_pecuaria         5


In [6]:
# Sample: Goiânia (município metropolitano grande, baseline conhecido)
g = data[data['geocode'] == '5208707'].iloc[0]
print('=== Goiânia (5208707) ===')
for c in ['municipio', 'uf', 'bioma', 'pop_2017', 'area_municipal_km2',
          'pib_percap', 'pam_area_cana', 'idhm_longevidade', 'idhm_educacao',
          'idhm_renda', 'gini',
          'densidade_pop', 'share_vab_agro', 'tratores_per_estab',
          'share_estab_af', 'log_pop', 'log_pib_percap']:
    if c in g.index:
        v = g[c]
        if isinstance(v, (int, np.integer)):
            print(f'  {c:30s}: {v}')
        elif isinstance(v, float):
            print(f'  {c:30s}: {v:.4f}')
        else:
            print(f'  {c:30s}: {v!r}')

=== Goiânia (5208707) ===
  municipio                     : 'Goiânia'
  uf                            : 'GO'
  bioma                         : 'Cerrado'
  pop_2017                      : 1466105
  area_municipal_km2            : 737.4500
  pib_percap                    : 33455
  pam_area_cana                 : 0.0000
  idhm_longevidade              : 0.8380
  idhm_educacao                 : 0.7390
  idhm_renda                    : 0.8240
  gini                          : 0.5800
  densidade_pop                 : 1988.0738
  share_vab_agro                : 0.0009
  tratores_per_estab            : 0.3948
  share_estab_af                : 0.4642
  log_pop                       : 14.1981
  log_pib_percap                : 10.4180


In [7]:
# Distribuições das principais covariáveis para o PSM
covs_principais = [
    'pib_percap', 'idhm_longevidade', 'idhm_educacao', 'idhm_renda', 'gini',
    'densidade_pop', 'share_vab_agro', 'tratores_per_estab',
    'share_estab_af', 'pam_area_cana',
]
data[covs_principais].describe().T

,count,mean,std,min,25%,50%,75%,max
pib_percap,2363.0,27135.026238,23651.291439,6087.000000,14449.500000,21590.000000,31872.500000,346739.000000
idhm_longevidade,2362.0,0.826279,0.025096,0.723000,0.809000,0.826000,0.844000,0.890000
idhm_educacao,2362.0,0.603931,0.077905,0.324000,0.552000,0.607000,0.661000,0.825000
idhm_renda,2362.0,0.682666,0.053051,0.502000,0.653000,0.688000,0.717000,0.891000
gini,2362.0,0.469691,0.057971,0.320000,0.430000,0.470000,0.500000,0.780000
densidade_pop,2363.0,149.785721,800.038785,0.304191,13.124216,26.983349,58.225819,15211.825689
share_vab_agro,2363.0,0.208736,0.155497,0.000000,0.081893,0.179991,0.316100,0.763299
tratores_per_estab,2358.0,0.646041,0.805181,0.000000,0.197114,0.456069,0.859046,19.529412
share_estab_af,2358.0,0.658115,0.139044,0.000000,0.573970,0.671552,0.757099,1.000000
pam_area_cana,2363.0,3836.537029,9191.616132,0.000000,2.000000,45.000000,2850.000000,99000.000000


## Auditoria — provenance das colunas

In [8]:
prov = result['provenance']
print(f'Total: {len(prov)} colunas processadas')
print(f'\nPor status:')
print(prov["status"].value_counts().to_string())
print()
print(f'\nTop 20 colunas descartadas (não estavam em COLUMN_RENAME):')
drop_sample = prov[prov["status"] == "descartada"].head(20)
drop_sample[['original']].to_string(index=False)

Total: 163 colunas processadas

Por status:
status
descartada    105
renomeada      58


Top 20 colunas descartadas (não estavam em COLUMN_RENAME):


'                      original\n                     0_ano_ref\n                      0_cd_reg\n                      0_nm_reg\n                       0_cd_uf\n                       0_nm_if\n                    1_impostos\n                   1_top1_vadc\n                   1_top2_vadc\n                   1_top3_vadc\n        3_mb_sharegrp_pre_agua\n     3_mb_sharegrp_pre_algodao\n        3_mb_sharegrp_pre_cafe\n      3_mb_sharegrp_pre_outros\n3_mb_sharegrp_pre_urbano_infra\n         4_area_colhida_ha_alg\n     4_area_colhida_ha_cafarab\n      4_area_colhida_ha_cafcan\n              4_quant_prod_alg\n          4_quant_prod_cafarab\n           4_quant_prod_cafcan'

## Resumo

Se output saiu com:
- 2.363 munis CS
- 71 colunas (58 renomeadas + 7 derivadas + 6 dummies bioma)
- Goiânia com IDHM longevidade ~0.84, PIB per capita ~33k
- Biomas corrigidos sem mojibake

PSM baseline está validado.

**Próximas camadas:**
- `08_coverage_audit.ipynb` — auditoria F4 final (cobertura todas fontes × munis × anos) — *mas opcional, F3 do SEEG já cobre o crítico*.
- `09_assembly.ipynb` — ⭐ painel completo: união de todos os outcomes (SEEG, SICAR), tratamento (ANP), covariáveis (PSM, PAM, MapBiomas), filtro canavieiro, painel balanceado pronto para CS/SDID/TWFE.

**Pipeline está praticamente completo.** Faltam apenas ajustes finais e o assembly.